# turboquant-pro — Reproducible Benchmark Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahb-sjsu/turboquant-pro/blob/master/notebooks/turboquant_benchmark.ipynb)

Reproduce turboquant-pro's embedding-compression results **end-to-end on public data**, in a few minutes, on CPU (Colab-friendly).

It compares compression ratio, retrieval **recall@10/@100**, and throughput against product quantization (PQ) and an exact baseline. On private 199k LaBSE embeddings turboquant-pro reaches **recall@10 ≈ 0.999 at ~30× vs PQ ≈ 0.57** — this notebook reproduces the same *pattern* on a public dataset so anyone can verify it.

In [ ]:
!pip install -q turboquant-pro faiss-cpu sentence-transformers datasets matplotlib pandas

## 1. Configure

In [ ]:
N = 20000          # corpus size (raise for a larger run)
QUERIES = 1000
OUT_DIM = 128      # PCA-Matryoshka target dimension
BITS = [2, 3, 4]   # TurboQuant bit-widths to sweep
OVERSAMPLE = 5     # rerank candidate multiplier
MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim encoder
SEED = 42

## 2. Build a real embedding corpus (public data)
Embeds AG-News headlines with a small sentence-transformer. Falls back to synthetic vectors if offline.

In [ ]:
import numpy as np

def load_real():
    from datasets import load_dataset
    from sentence_transformers import SentenceTransformer
    ds = load_dataset("ag_news", split=f"train[:{N}]")
    texts = [r["text"] for r in ds]
    enc = SentenceTransformer(MODEL)
    e = enc.encode(texts, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
    return np.asarray(e, dtype=np.float32)

try:
    X = load_real()
    print("real embeddings:", X.shape)
except Exception as ex:
    print("offline -> synthetic fallback:", ex)
    rng = np.random.default_rng(SEED)
    X = rng.standard_normal((N, 384)).astype(np.float32)
    X /= np.linalg.norm(X, axis=1, keepdims=True) + 1e-30

Q, C = X[:QUERIES], X[QUERIES:]
dim = X.shape[1]
print("corpus:", C.shape, " queries:", Q.shape, " dim:", dim)

## 3. Exact ground truth + metrics

In [ ]:
def exact_topk(Qq, Xx, k):
    out = np.zeros((len(Qq), k), dtype=np.int64)
    for s in range(0, len(Qq), 256):
        sims = Qq[s:s+256] @ Xx.T
        idx = np.argpartition(-sims, k, axis=1)[:, :k]
        for r in range(idx.shape[0]):
            idx[r] = idx[r][np.argsort(-sims[r, idx[r]])]
        out[s:s+256] = idx
    return out

def recall(gt, ap, k):
    return float(np.mean([len(set(gt[i, :k]) & set(ap[i, :k])) / k for i in range(len(gt))]))

gt = exact_topk(Q, C, 100)
print("ground truth ready")

## 4. Run the benchmark

In [ ]:
import time, faiss
from turboquant_pro import PCAMatryoshka

rows = []
def add(method, bpv, qps, r10, r100, note=""):
    rows.append(dict(method=method, bytes_per_vec=round(bpv, 1),
                     compression_x=round(dim * 4 / bpv, 1), qps=round(qps, 1),
                     recall_at_10=round(r10, 4),
                     recall_at_100=(round(r100, 4) if r100 == r100 else None), note=note))

# exact fp32 baseline
flat = faiss.IndexFlatIP(dim); flat.add(C)
t = time.time(); _, I = flat.search(Q, 100)
add("fp32-flat (exact)", dim * 4, QUERIES / (time.time() - t), recall(gt, I, 10), recall(gt, I, 100))

# product-quantization baselines at matched byte budgets
for bits in BITS:
    m = max(1, (OUT_DIM * bits) // 8)
    while dim % m:
        m -= 1
    pq = faiss.IndexPQ(dim, m, 8); pq.train(C[:min(len(C), 50000)]); pq.add(C)
    t = time.time(); _, I = pq.search(Q, 100)
    add(f"PQ (m={m})", m, QUERIES / (time.time() - t), recall(gt, I, 10), recall(gt, I, 100), f"~{bits}-bit budget")

# turboquant-pro: PCA-Matryoshka + TurboQuant, single-stage and +rerank
for bits in BITS:
    pca = PCAMatryoshka(input_dim=dim, output_dim=OUT_DIM); pca.fit(C[:min(len(C), 50000)])
    pipe = pca.with_quantizer(bits=bits)
    codes = pipe.compress_batch(C)
    recon = np.asarray(pipe.decompress_batch(codes), dtype=np.float32)
    recon /= np.linalg.norm(recon, axis=1, keepdims=True) + 1e-30
    rdim = recon.shape[1]
    idx = faiss.IndexFlatIP(rdim); idx.add(recon)
    Qs = Q if rdim == dim else np.asarray(pca.transform(Q), dtype=np.float32)
    Qs = Qs / (np.linalg.norm(Qs, axis=1, keepdims=True) + 1e-30)
    try:
        bpv = float(pipe.estimate_storage(len(C))) / len(C)
    except Exception:
        bpv = OUT_DIM * bits / 8 + 4
    t = time.time(); _, I1 = idx.search(Qs, 100)
    add(f"turboquant-pro TQ{bits} (1-stage)", bpv, QUERIES / (time.time() - t), recall(gt, I1, 10), recall(gt, I1, 100))
    kk = 10 * OVERSAMPLE
    t = time.time(); _, cand = idx.search(Qs, kk)
    rr = np.zeros((QUERIES, 10), dtype=np.int64)
    for i in range(QUERIES):
        c = cand[i]; sc = C[c] @ Q[i]; rr[i] = c[np.argsort(-sc)[:10]]
    add(f"turboquant-pro TQ{bits} (+rerank x{OVERSAMPLE})", bpv, QUERIES / (time.time() - t),
        recall(gt, rr, 10), float("nan"), "rerank uses fp32 originals")

print("done")

## 5. Results table

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
df

## 6. Compression–recall frontier
Up and to the right is better: more compression at higher recall.

In [ ]:
import matplotlib.pyplot as plt

pq = sorted((r["compression_x"], r["recall_at_10"]) for r in rows if r["method"].startswith("PQ"))
tq = sorted((r["compression_x"], r["recall_at_10"]) for r in rows if "rerank" in r["method"])
fig, ax = plt.subplots(figsize=(7, 5))
if pq:
    ax.plot(*zip(*pq), "x--", ms=10, label="PQ")
if tq:
    ax.plot(*zip(*tq), "o-", ms=8, label="turboquant-pro (+rerank)")
ax.set_xlabel("compression x"); ax.set_ylabel("recall@10")
ax.set_title("Compression vs recall@10"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## Takeaway

At matched compression, **turboquant-pro dominates PQ on retrieval recall**. The private-data result (199k LaBSE: recall@10 ≈ 0.999 vs PQ ≈ 0.57 at ~30×) is in [`benchmarks/RESULTS_labse_199k.md`](../benchmarks/RESULTS_labse_199k.md); this notebook reproduces the same pattern on public AG-News embeddings so anyone can verify it on demand.